In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import glob
from scipy.stats import linregress
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# select demand for victoria and sort in order of date
demand_vic = glob.glob("/Users/Coder/Desktop/ADS1002-project/data/raw/demand/*_TAS1.csv")
demand_vic.sort()

# read demand data files and combine
df_list = [pd.read_csv(f) for f in demand_vic]
if df_list:
    demand_df = pd.concat(df_list, ignore_index=True)
    print("CSV files loaded")
else:
    print("No CSV files were loaded.")

# rename and drop columns
demand_df = demand_df.rename(columns={
    "SETTLEMENTDATE": "Datetime",
    "TOTALDEMAND": "Total Demand",
    "REGION": "State"
})
demand_df = demand_df.drop(columns=["PERIODTYPE"])

# Convert Datetime to desired format, handling mixed formats
demand_df['Datetime'] = pd.to_datetime(demand_df['Datetime'], format='mixed', dayfirst=True).dt.strftime('%Y/%m/%d %H:%M:%S')

# Convert Datetime to desired format
demand_df['Datetime'] = pd.to_datetime(demand_df['Datetime'], format='%Y/%m/%d %H:%M:%S').dt.strftime('%Y/%m/%d %H:%M')

print(demand_df)


CSV files loaded
       REGION       SETTLEMENTDATE  TOTALDEMAND      RRP PERIODTYPE
0        TAS1  2005/05/16 14:00:00   1277.46000   833.33      TRADE
1        TAS1  2005/05/16 14:30:00   1279.19667  4500.00      TRADE
2        TAS1  2005/05/16 15:00:00   1282.52500  4500.00      TRADE
3        TAS1  2005/05/16 15:30:00   1281.69167  8166.67      TRADE
4        TAS1  2005/05/16 16:00:00   1287.23500  4500.00      TRADE
...       ...                  ...          ...      ...        ...
256432   TAS1  2019/12/31 22:00:00   1032.56000    65.31      TRADE
256433   TAS1  2019/12/31 22:30:00   1032.85000    80.97      TRADE
256434   TAS1  2019/12/31 23:00:00   1024.17000    73.03      TRADE
256435   TAS1  2019/12/31 23:30:00   1008.68000    81.76      TRADE
256436   TAS1  2020/01/01 00:00:00   1006.70000    93.88      TRADE

[256437 rows x 5 columns]


In [77]:
#All data is loaded below, next step is to clean the data
# Check for missing values
missing_values = demand_df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

# None of data has missing values, so we can proceed to the next step

Missing values in each column:
State           0
Datetime        0
Total Demand    0
RRP             0
dtype: int64


In [95]:
# Print number of data points where TOTALDEMAND is under 0
negative_demand_count = len(demand_df[demand_df['Total Demand'] < 0])
print(f"Number of data points where TOTALDEMAND is negative: {negative_demand_count}")

negative_RRP = len(demand_df[demand_df['RRP'] < 0])
print(f"Number of data points where TOTALDEMAND is negative: {negative_RRP}")

# Find the minimum and maximum RRP
min_rrp = demand_df['RRP'].min()
max_rrp = demand_df['RRP'].max()

# Print the results
print(f"Minimum RRP: {min_rrp}")
print(f"Maximum RRP: {max_rrp}")

Number of data points where TOTALDEMAND is negative: 0
Number of data points where TOTALDEMAND is negative: 1820
Minimum RRP: -1000.0
Maximum RRP: 12400.26


In [79]:
# read weather data for Tasmania
weather_df = pd.read_csv(r"C:\Users\Coder\Desktop\ADS1002-project\Data\raw\weather\HM01X_Data_094029_999999999743964.txt")
# print(weather_df)
#print(weather_df.columns)

# combine datetime into one variable with the correct format
weather_df['Datetime'] = pd.to_datetime({
    'year': weather_df['Year Month Day Hour Minutes in YYYY'],
    'month': weather_df['MM'],
    'day': weather_df['DD'],
    'hour': weather_df['HH24'],
    'minute': weather_df['MI format in Local time']
})
weather_df['Datetime'] = weather_df['Datetime'].dt.strftime('%Y/%m/%d %H:%M')

# rename columns
weather_df = weather_df.rename(columns={
    "Precipitation since 9am local time in mm": "Precipitation (mm)",
    "Air Temperature in degrees C": "Air Temp (C)",
    "Relative humidity in percentage %": "Humidity (%)",
    "Wind speed in km/h": "Wind Speed (km/h)"
})

# convert variables to int
cols_to_convert = ["Precipitation (mm)", "Air Temp (C)", "Humidity (%)", "Wind Speed (km/h)"]

weather_df[cols_to_convert] = weather_df[cols_to_convert].apply(
    pd.to_numeric, errors='coerce'
)

print(weather_df.columns)
print(weather_df.head())

C:\Users\Coder\AppData\Local\Temp\ipykernel_20492\3781307506.py:2: DtypeWarning: Columns (12,14,18,20,22,24,26,28,32) have mixed types. Specify dtype option on import or set low_memory=False.
  weather_df = pd.read_csv(r"C:\Users\Coder\Desktop\ADS1002-project\Data\raw\weather\HM01X_Data_094029_999999999743964.txt")


Index(['hm', 'Station Number', 'Year Month Day Hour Minutes in YYYY', 'MM',
       'DD', 'HH24', 'MI format in Local time',
       'Year Month Day Hour Minutes in YYYY.1', 'MM.1', 'DD.1', 'HH24.1',
       'MI format in Local standard time', 'Precipitation (mm)',
       'Quality of precipitation since 9am local time', 'Air Temp (C)',
       'Quality of air temperature', 'Wet bulb temperature in degrees C',
       'Quality of Wet bulb temperature', 'Dew point temperature in degrees C',
       'Quality of dew point temperature', 'Humidity (%)',
       'Quality of relative humidity', 'Wind Speed (km/h)',
       'Wind speed quality', 'Wind direction in degrees true',
       'Wind direction quality',
       'Speed of maximum windgust in last 10 minutes in  km/h',
       'Quality of speed of maximum windgust in last 10 minutes',
       'Mean sea level pressure in hPa', 'Quality of mean sea level pressure',
       'Station level pressure in hPa', 'Quality of station level pressure',
       'AW

In [94]:
# merge demand and weather data
mergedvic_df = pd.merge(
    demand_df,
    weather_df[["Datetime", "Precipitation (mm)", "Air Temp (C)", "Humidity (%)", "Wind Speed (km/h)"]],
    on='Datetime',
    how='inner'
)

# precip is only recorded after 9am
# fill not recorded values as NR for ease of use during analysis and modelling
mergedvic_df["Precipitation (mm)"] = mergedvic_df["Precipitation (mm)"].fillna("NR")

# drop invalid and empty values
mergedvic_df = mergedvic_df.drop_duplicates()
mergedvic_df = mergedvic_df.dropna()

print(mergedvic_df)

# save cleaned data to new file
mergedvic_df.to_csv("TAS_Processed_data.csv", index=False)


       State          Datetime  Total Demand      RRP Precipitation (mm)  \
0       TAS1  2005/05/16 14:00    1277.46000   833.33                0.0   
1       TAS1  2005/05/16 14:30    1279.19667  4500.00                0.0   
2       TAS1  2005/05/16 15:00    1282.52500  4500.00                0.0   
3       TAS1  2005/05/16 15:30    1281.69167  8166.67                0.0   
4       TAS1  2005/05/16 16:00    1287.23500  4500.00                0.0   
...      ...               ...           ...      ...                ...   
254959  TAS1  2019/12/31 22:00    1032.56000    65.31                0.0   
254960  TAS1  2019/12/31 22:30    1032.85000    80.97                0.0   
254961  TAS1  2019/12/31 23:00    1024.17000    73.03                0.0   
254962  TAS1  2019/12/31 23:30    1008.68000    81.76                0.0   
254963  TAS1  2020/01/01 00:00    1006.70000    93.88                0.0   

        Air Temp (C)  Humidity (%)  Wind Speed (km/h)  
0               12.1          6